# Tutorial: Finding disease-associated cell neighborhoods with scEPS

[scEPS](https://github.com/Genentech/sceps) (single-cell Expression exPlainability
Statistics) combines GWAS statistics with a multi-donor single-cell atlas to answer one
question: **in which cell neighborhoods does variation in disease-gene expression explain
variation in the disease phenotype across donors?**

A *cell neighborhood* is a local region of the k-NN graph, anchored on a single cell. In
each neighborhood, scEPS decomposes the variance of the donor-level phenotype into the part
attributable to GWAS genes, the part attributable to a set of **mean-expression-matched
control genes**, and the part attributable to all remaining genes. This gives, per
neighborhood:

$$
d = \omega^2_{gwas} - \omega^2_{ctrl}
$$

where $\omega^2_{gwas}$ is the phenotypic variance explained *per GWAS gene* and
$\omega^2_{ctrl}$ the same quantity per control gene. The control set matters: highly
expressed genes simply have more expression variance to offer, so comparing GWAS genes
against expression-matched controls is what makes $d$ interpretable. A positive,
significant $d$ says the neighborhood's disease-gene expression tracks the phenotype more
closely than chance expression variation of comparable magnitude would.

The method is described in the [preprint](https://www.medrxiv.org/content/10.64898/2026.06.26.26356714v1);
results for the diseases in the paper can be browsed in the
[result explorer](https://scepsresultexplorer.streamlit.app/).

## The design of this tutorial

We run scEPS on [OneK1K](https://onek1k.org/) — 1.25M PBMCs from 981 donors — for
**rheumatoid arthritis (RA)**, an immune-mediated disease for which PBMCs are the relevant
tissue.

OneK1K is a healthy population cohort, so it carries no RA case/control status to
decompose. The scEPS paper handles exactly this situation by using a **polygenic risk score
as the phenotype**, and we follow that design: the phenotype is an RA PRS computed from the
donors' own genotypes, and the GWAS genes come from an RA GWAS.

```{important}
A PRS is a linear function of genotype. That has a consequence worth keeping in view:
$\omega^2_{gwas}$ can pick up *cis*-eQTL genotype→expression signal — genotype predicting
its own nearby expression — rather than disease-relevant expression variation. A PRS
phenotype is a stand-in for a measured phenotype, not an equivalent of one. If you take
this design further than a tutorial, re-running Step 1 with gene-level statistics for an
unrelated trait is the standard control: signal specific to the trait of interest should
not reproduce there.
```

```{warning}
This is a tutorial, not a finding. A single PRS-as-phenotype analysis on a healthy cohort,
restricted to a subsample of donors, cells and neighborhoods, is not evidence about RA
biology. Read the numbers below as a demonstration of the mechanics and the output formats.
```

## The workflow

scEPS ships as four command-line tools, one per step, preceded by two preparation steps
you do yourself:

| Step | Tool | What it produces |
| --- | --- | --- |
| 0a | MAGMA (via cellink) | gene-level association statistics (`GENE`, `ZSTAT`) per trait |
| 0b | PLINK (via cellink) | the donor phenotype — an RA polygenic risk score |
| 0c | scanpy | a QC'd `.h5ad` with a batch-harmonized k-NN graph |
| 1 | `sceps` | $d$ and the $\omega^2$ components for every cell neighborhood |
| 2 | `sceps-cluster-neighborhood` | approximately independent neighborhood blocks, for the block bootstrap |
| 3 | `sceps-aggregate` | average $d$ per cell type, with block-bootstrap standard errors |
| 4 | `sceps-corr` | correlation between $d$ and each gene's expression across neighborhoods |

The [scEPS wiki](https://github.com/Genentech/sceps/wiki) is the reference for every option;
this tutorial covers the ones you need to get a first analysis out.

## Setup

scEPS is on PyPI and installs its own command-line tools:

```shell
pip install sceps
```

Two external tools are also needed. **PLINK** must be on `PATH` (`conda install -c bioconda
plink`) — it computes the PRS and the genotype PCs. **MAGMA** is downloaded by the notebook.
The `liftover` extra (`pip install "cellink-tools[datasets]"`) is needed by some cellink
genotype paths.

```{important}
Everything this notebook writes — a 4.4 GB expression file, a genotype VCF, PLINK binaries,
MAGMA output, scEPS results — goes to `WORK_DIR`, **not** into the repository. Point it at a
scratch filesystem with ~40 GB free, either by editing the cell below or by setting
`SCEPS_TUTORIAL_WORKDIR`.
```

In [ ]:
import gzip
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import yaml

import cellink
from cellink.resources import (
    get_gwas_catalog_study_summary_stats,
    get_onek1k,
    get_pgs_catalog_score,
    get_pgs_catalog_score_file,
)
from cellink.tl.external import run_magma_pipeline

sc.settings.verbosity = 1

# ---------------------------------------------------------------------------
# Everything you might want to change lives here.
# ---------------------------------------------------------------------------
SEED = 0

WORK_DIR = Path(os.environ.get("SCEPS_TUTORIAL_WORKDIR", "~/sceps_tutorial_work")).expanduser()
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- Step 0a: the RA GWAS. European-only, matching OneK1K's ancestry.
RA_GWAS = "GCST90132223"  # Ishigaki 2022, RA: 22,350 cases / 74,823 controls
RA_GWAS_N = 22_350 + 74_823
GWAS_BUILD = "GRCh37"

# --- Step 0b: the PRS. Privé 2022, LDpred2 on UK Biobank, 95,083 variants, European.
RA_PGS_ID = "PGS002088"
N_GENO_PCS = 10  # genotype PCs available as covariates

# --- Step 0c: how much of the atlas to keep. OneK1K is 1.25M cells from 981 donors;
# scEPS analyzes one neighborhood per cell and its regression has one row per donor
# *pair*, so a tutorial-sized subsample of both axes is essential.
N_DONORS = 100  # donors kept (scEPS needs >= 20)
N_TOP_CELLTYPES = 6  # most abundant cell types, so Step 3 has enough per type
N_CELLS = 5_000  # cells kept, spread across all retained donors
N_HVG = 2_000  # highly variable genes, flagged for reference (the graph uses X_harmony)
N_NEIGHBORS = 15

# --- Step 1: the phenotype and the donor-level covariates to adjust for.
PHENO = "ra_prs"
COVAR = "sex,geno_pc1,geno_pc2,geno_pc3,sceps.prop_cells_local"
N_NEIGHBORHOODS = 300  # neighborhoods analyzed, of N_CELLS available
N_PROBE = 20  # neighborhoods in the timing probe run

# --- Step 2 / Step 4
N_KMEANS_CLUSTERS = 20  # approximately independent blocks (default 50 assumes more cells)
MIN_NUM_NONZERO = 50  # a gene must be expressed in this many analyzed neighborhoods

# --- Paths. Nothing is written inside the repository.
DATA_HOME = WORK_DIR / "cellink_data"
GENO_DIR = DATA_HOME / "onek1k"
ADATA_PATH = WORK_DIR / "sceps_input.h5ad"
OUT_DIR = WORK_DIR / "sceps_out"
OUT_DIR.mkdir(exist_ok=True)

# configs/magma.yaml ships in the cellink repository, not in the installed package, so
# look for it relative to the notebook and relative to a source checkout.
_magma_config_candidates = [
    Path("../../configs/magma.yaml"),
    Path(cellink.__file__).parents[2] / "configs" / "magma.yaml",
]
MAGMA_CONFIG = next((str(p) for p in _magma_config_candidates if p.exists()), None)
if MAGMA_CONFIG is None:
    raise FileNotFoundError(
        "configs/magma.yaml not found; clone https://github.com/theislab/cellink and point "
        f"MAGMA_CONFIG at its configs/magma.yaml. Looked in: {_magma_config_candidates}"
    )

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print("WORK_DIR:    ", WORK_DIR)
print("MAGMA_CONFIG:", MAGMA_CONFIG)

Confirm the scEPS command-line tools and PLINK are available:

In [ ]:
!sceps --help | head -8
!plink --version

## Step 0a: gene-level GWAS statistics

scEPS needs to know which genes are implicated by the GWAS, and how strongly. It takes this
as a whitespace-delimited text file with two columns it actually reads:

```text
GENE       ZSTAT
```

`GENE` must use the **same identifiers as the single-cell data** — Ensembl IDs here, since
that is what OneK1K's `var_names` are — and `ZSTAT` is the gene-level association
statistic. Everything else in the file is ignored, so MAGMA's output can be passed straight
through.

[MAGMA](https://cncr.nl/research/magma/) produces this in two steps: map SNPs to genes
within a window, then aggregate SNP-level p-values per gene. cellink's
`run_magma_pipeline` wraps both, downloading the 1000 Genomes EUR reference panel and the
gene location file for you.

First the MAGMA binary itself, which is not pip-installable:

In [ ]:
import io
import platform
import stat
import zipfile

import requests

MAGMA_DIR = WORK_DIR / "magma"
MAGMA_BIN = MAGMA_DIR / "magma"

if not MAGMA_BIN.exists():
    share_ids = {"Linux": "lxDgt2dNdNr6DYt", "Darwin": "1M1d9vHtVidEwvU"}
    system = platform.system()
    if system not in share_ids:
        raise RuntimeError(f"No known MAGMA v1.10 build for {system!r}; see https://cncr.nl/research/magma/")
    url = f"https://vu.data.surf.nl/public.php/dav/files/{share_ids[system]}"
    MAGMA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(requests.get(url, timeout=600).content)) as zf:
        zf.extractall(MAGMA_DIR)
    MAGMA_BIN.chmod(MAGMA_BIN.stat().st_mode | stat.S_IEXEC)

print(subprocess.run([str(MAGMA_BIN), "--version"], capture_output=True, text=True).stdout.strip()[:200])

`get_gwas_catalog_study_summary_stats` fetches harmonized summary statistics from the GWAS
Catalog; `run_magma_pipeline` turns them into gene-level statistics. We use
[GCST90132223](https://www.ebi.ac.uk/gwas/studies/GCST90132223) — Ishigaki et al. (2022),
22,350 cases and 74,823 controls, European-only, which matches OneK1K's ancestry. The
download is large and MAGMA's gene analysis takes a while, so the result is cached on disk
and skipped on re-runs.

In [ ]:
# MAGMA wants the GWAS Catalog's harmonized column names mapped to its own.
COL_MAPPING = {"chromosome": "CHR", "base_pair_location": "BP", "p_value": "P"}

MAGMA_PREFIX = WORK_DIR / "magma_RA"
magma_genes = Path(f"{MAGMA_PREFIX}.genes.out")

if magma_genes.exists():
    print(f"{magma_genes.name} already exists, skipping")
else:
    gwas = get_gwas_catalog_study_summary_stats(RA_GWAS, genome_build=GWAS_BUILD)
    print(f"{gwas.shape[0]:,} variants")

    magma_genes = run_magma_pipeline(
        gwas,
        output_prefix=str(MAGMA_PREFIX),
        col_mapping=COL_MAPPING,
        n_samples=RA_GWAS_N,
        genome_build=GWAS_BUILD,
        gene_id_type="ensembl",  # match OneK1K's var_names
        ld_source="reference_panel",
        reference_panel="EUR",
        config_file=MAGMA_CONFIG,
        magma_bin=str(MAGMA_BIN),
    )

print("gene-level statistics:", magma_genes)

MAGMA's output, and the two columns scEPS reads:

In [ ]:
genes = pd.read_csv(magma_genes, sep=r"\s+")
print(f"{genes.shape[0]:,} genes")
print(genes.sort_values("P").head(6)[["GENE", "CHR", "NSNPS", "ZSTAT", "P"]].to_string(index=False))

## Step 0b: the phenotype — an RA polygenic risk score

OneK1K has no disease phenotype, so we build one from the donors' genotypes, following the
paper's design. Three ingredients: the genotypes, a published set of PRS weights, and PLINK
to combine them.

### The genotypes

`get_onek1k()` returns a `DonorData` with both modalities, but building the genotype side
means a `vcf2zarr` pass over a genome-wide VCF plus PLINK filtering, pruning and kinship —
and a PRS needs none of that. So we pass `only_download=True`, which fetches the files and
skips all preprocessing, then run just the two PLINK steps a PRS actually requires.

Reading cellink's own config rather than hard-coding URLs means this keeps working if they
change.

In [ ]:
onek1k_config = Path(cellink.__file__).parent / "resources" / "config" / "onek1k.yaml"
print("files get_onek1k fetches:", [f["filename"] for f in yaml.safe_load(onek1k_config.read_text())["remote_files"]])

# only_download=True skips the genotype preprocessing (and so needs no vcf2zarr).
# Files already present are verified by checksum and skipped.
get_onek1k(data_home=str(DATA_HOME), only_download=True)

for p in sorted(GENO_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:45} {p.stat().st_size / 1e9:6.2f} GB")

Convert the VCF to PLINK binary format. cellink's own `preprocess_vcf_to_plink` helper does
this too, but it overwrites the variant IDs with `chrom:pos_ref_alt`; we keep the IDs the
VCF carries, because matching PGS Catalog weights by rsID is the most robust option when
available.

In [ ]:
GENO_PREFIX = GENO_DIR / "onek1k_prs"

if not Path(f"{GENO_PREFIX}.bed").exists():
    subprocess.run(
        [
            "plink",
            "--threads",
            "10",
            "--keep-allele-order",
            "--const-fid",
            "--vcf",
            str(GENO_DIR / "OneK1K.noGP.vcf.gz"),
            "--maf",
            "0.01",
            "--make-bed",
            "--out",
            str(GENO_PREFIX),
        ],
        check=True,
    )

bim = pd.read_csv(f"{GENO_PREFIX}.bim", sep=r"\s+", header=None, names=["chrom", "id", "cm", "pos", "a1", "a2"])
fam = pd.read_csv(f"{GENO_PREFIX}.fam", sep=r"\s+", header=None)
frac_rsid = float(bim["id"].astype(str).str.startswith("rs").mean())

print(f"{bim.shape[0]:,} variants x {fam.shape[0]:,} donors")
print("variant ID examples:", bim["id"].head(3).tolist())
print(f"fraction of IDs that are rsIDs: {frac_rsid:.3f}")

### The PRS weights

We use [PGS002088](https://www.pgscatalog.org/score/PGS002088/) — an RA score from Privé et
al. (2022), built with LDpred2 on UK Biobank, 95,083 variants, European ancestry. PGS
Catalog publishes both the author-submitted file and *harmonized* files whose positions are
lifted to a named build, which is what makes matching reliable.

In [ ]:
pgs_meta = get_pgs_catalog_score(RA_PGS_ID)
print(f"{pgs_meta['id']}: {pgs_meta['trait_reported']}")
print(f"  variants:   {pgs_meta['variants_number']:,}")
print(f"  build:      {pgs_meta['variants_genomebuild']}")
print(f"  weights:    {pgs_meta['weight_type']}")
print(f"  reference:  {pgs_meta['publication']['firstauthor']} " f"{pgs_meta['publication']['date_publication'][:4]}")

pgs_path = get_pgs_catalog_score_file(RA_PGS_ID, dest=WORK_DIR / f"{RA_PGS_ID}.txt.gz", return_path=True)

with gzip.open(pgs_path, "rt") as fh:
    header = [ln.rstrip() for ln in fh if ln.startswith("#")]
print("\n".join(header[:12]))

pgs = pd.read_csv(pgs_path, sep="\t", comment="#")
print("\ncolumns:", list(pgs.columns))
print(pgs.head(3).to_string(index=False))

Build the three-column file PLINK's `--score` wants — variant ID, effect allele, weight —
matching on whichever key the genotypes actually support. How many variants carry over is
the number to watch: a score that overlaps almost nothing produces a meaningless PRS.

In [ ]:
snp_col = "rsID" if ("rsID" in pgs.columns and frac_rsid > 0.5) else None

if snp_col is not None:
    score_df = pgs[[snp_col, "effect_allele", "effect_weight"]].copy()
    key_desc = "rsID"
else:
    # Fall back to chrom:pos, which requires the builds to agree.
    score_df = pgs.copy()
    score_df["variant_key"] = score_df["chr_name"].astype(str) + ":" + score_df["chr_position"].astype(int).astype(str)
    bim_key = bim["chrom"].astype(str) + ":" + bim["pos"].astype(str)
    id_by_key = dict(zip(bim_key, bim["id"].astype(str), strict=True))
    score_df[snp_col := "variant_id"] = score_df["variant_key"].map(id_by_key)
    score_df = score_df.dropna(subset=[snp_col])[[snp_col, "effect_allele", "effect_weight"]]
    key_desc = "chrom:pos"

score_df.columns = ["SNP", "A1", "WEIGHT"]
n_overlap = score_df["SNP"].isin(set(bim["id"].astype(str))).sum()
print(f"matching on {key_desc}: {n_overlap:,} of {pgs.shape[0]:,} score variants are in the genotypes")

SCORE_FILE = WORK_DIR / f"{RA_PGS_ID}.plink_score.txt"
score_df.to_csv(SCORE_FILE, sep="\t", index=False)
score_df.head()

### Scoring, and the genotype PCs

`plink --score` sums the dosage-weighted effects per donor. We also compute genotype
principal components from LD-pruned variants: a PRS correlates with ancestry, so the PCs
belong in the covariate list even in a cohort that is nominally one ancestry.

In [ ]:
PRS_PREFIX = WORK_DIR / "onek1k_ra_prs"
if not Path(f"{PRS_PREFIX}.profile").exists():
    subprocess.run(
        [
            "plink",
            "--threads",
            "10",
            "--bfile",
            str(GENO_PREFIX),
            "--score",
            str(SCORE_FILE),
            "1",
            "2",
            "3",
            "header",
            "sum",
            "--out",
            str(PRS_PREFIX),
        ],
        check=True,
    )

PCA_PREFIX = WORK_DIR / "onek1k_geno_pcs"
if not Path(f"{PCA_PREFIX}.eigenvec").exists():
    subprocess.run(
        [
            "plink",
            "--threads",
            "10",
            "--bfile",
            str(GENO_PREFIX),
            "--maf",
            "0.05",
            "--indep-pairwise",
            "200",
            "50",
            "0.25",
            "--out",
            str(PCA_PREFIX),
        ],
        check=True,
    )
    subprocess.run(
        [
            "plink",
            "--threads",
            "10",
            "--bfile",
            str(GENO_PREFIX),
            "--extract",
            f"{PCA_PREFIX}.prune.in",
            "--pca",
            str(N_GENO_PCS),
            "--out",
            str(PCA_PREFIX),
        ],
        check=True,
    )

print(open(f"{PRS_PREFIX}.log").read()[-700:])

In [ ]:
prs = pd.read_csv(f"{PRS_PREFIX}.profile", sep=r"\s+")
score_col = "SCORESUM" if "SCORESUM" in prs.columns else "SCORE"
prs = prs.rename(columns={"IID": "donor_id", score_col: "ra_prs"})[["donor_id", "ra_prs", "CNT"]]

pcs = pd.read_csv(f"{PCA_PREFIX}.eigenvec", sep=r"\s+", header=None)
pcs = pcs.drop(columns=[0]).rename(columns={1: "donor_id"})
pcs.columns = ["donor_id"] + [f"geno_pc{i}" for i in range(1, pcs.shape[1])]

donor_pheno = prs.merge(pcs, on="donor_id")
donor_pheno["ra_prs_z"] = (donor_pheno["ra_prs"] - donor_pheno["ra_prs"].mean()) / donor_pheno["ra_prs"].std()

print(f"{donor_pheno.shape[0]:,} donors scored, {int(prs['CNT'].iloc[0]):,} variants used per donor")
print(donor_pheno[["ra_prs", "ra_prs_z", "geno_pc1", "geno_pc2"]].describe().loc[["mean", "std", "min", "max"]])
donor_pheno.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(donor_pheno["ra_prs_z"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].set(xlabel="RA PRS (standardized)", ylabel="donors", title=f"{RA_PGS_ID} across OneK1K donors")
axes[1].scatter(donor_pheno["geno_pc1"], donor_pheno["ra_prs_z"], s=12, alpha=0.6, color="#4C72B0")
axes[1].set(xlabel="genotype PC1", ylabel="RA PRS (standardized)", title="PRS vs ancestry PC1")
plt.tight_layout()
plt.show()

A roughly normal PRS distribution is what a genome-wide score should give. Any trend
against PC1 is the ancestry confounding that the `geno_pc*` covariates are there to absorb;
scEPS regresses covariates out of the phenotype before decomposing it.

## Step 0c: prepare the single-cell data

scEPS reads one `.h5ad` file and expects the following from it:

| Requirement | Why | Here |
| --- | --- | --- |
| ≥ 20 donors with phenotype variation | the variance decomposition is *across donors* | 981 donors, subsampled to 100 |
| a donor ID column in `adata.obs` | groups cells by donor | `donor_id` |
| a donor-level phenotype | the quantity whose variance is decomposed | `ra_prs` from Step 0b |
| a k-NN graph in `adata.obsp["connectivities"]` | defines the neighborhoods | built below |
| a batch-harmonized cell embedding | the graph should be built on it, and Step 2 clusters neighborhoods in it | `X_harmony`, shipped with the dataset |
| a cell type column in `adata.obs` | Step 3 aggregates by it | `celltype` |

Gene and cell identifiers can be left to the index: with `--gene-id-col` / `--cell-id-col`
unset, scEPS uses `adata.var.index` / `adata.obs.index` and calls them `sceps.gene_index` /
`sceps.cell_index`. We rely on that throughout, so the four steps agree on identifiers
without extra bookkeeping.

### The full preprocessing recipe (explained)

scEPS assumes QC'd, batch-harmonized data. The scEPS repository ships an example recipe in
`misc/preprocess_scdata.py`, reproduced below; it is *not* executed here because OneK1K
already comes QC'd with a harmonized embedding, and QC thresholds are dataset-specific.

In [ ]:
# --- misc/preprocess_scdata.py, for reference (not executed) ---------------
"""
import numpy as np
import scanpy as sc
import scanpy.external as sce

adata = sc.read_h5ad("<raw counts>.h5ad")

# Drop batches with too few cells
batch_nm = "<batch column>"
counts = adata.obs[batch_nm].value_counts()
adata = adata[adata.obs[batch_nm].isin(counts[counts >= 3].index)]

# QC and log-normalization
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata = adata[adata.obs.pct_counts_mt < 5, :]
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Batch-harmonized embedding and k-NN graph
sc.pp.highly_variable_genes(adata, n_top_genes=2000, inplace=True)
sc.pp.pca(adata, use_highly_variable=True)
sce.pp.harmony_integrate(adata, batch_nm, basis="X_pca", max_iter_harmony=50)
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata)
adata.write_h5ad("harmonized.h5ad")

# Optionally also save a copy with batch regressed out of the expression itself,
# and a copy with no adata.X at all (Steps 2 and 3 only need the graph, so the
# small file keeps their memory use down).
sc.pp.regress_out(adata, keys=[batch_nm])
adata.X = adata.X.astype(np.float32)
adata.write_h5ad("harmonized.regressout.h5ad")

del adata.X
adata.write_h5ad("harmonized.noX.h5ad")
"""

```{tip}
That recipe writes three files on purpose. Step 1 needs the expression matrix, but Steps 2
and 3 only need the graph and `obs`, so pointing them at the `.noX` copy is the easiest way
to cut memory on a large atlas.
```

### Subsampling

Two axes need subsampling. Cells, because scEPS analyzes one neighborhood per cell; and
**donors**, because the method-of-moments regression has one row per donor *pair* — 100
donors give 4,950 rows, but all 981 would give 481,690, far slower per neighborhood without
teaching anything extra. We also keep only the most abundant cell types, so Step 3 has
enough neighborhoods per type to aggregate over.

The atlas is 1.25M cells, so we open it in backed mode and pull only the subsample into
memory.

In [ ]:
full = sc.read_h5ad(GENO_DIR / "onek1k_cellxgene.h5ad", backed="r")
celltype_key = "predicted.celltype.l2"
obs = full.obs

print(full)
print(
    "\ndonors:",
    obs["donor_id"].nunique(),
    "| batches:",
    obs["pool_number"].nunique(),
    "| cell types:",
    obs[celltype_key].nunique(),
)
print("disease label:", obs["disease"].value_counts().to_dict())

The donor IDs in the expression file and in the genotypes are formatted differently;
`get_onek1k` reconciles them with `"OneK1K_" + donor_id.split("_")[1]` when it builds the
`DonorData`, and we apply the same rule so the PRS joins correctly.

In [ ]:
expr_donors = pd.Index(obs["donor_id"].astype(str).unique())
print("expression donor IDs:", expr_donors[:3].tolist())
print("genotype donor IDs:  ", donor_pheno["donor_id"].astype(str).head(3).tolist())

# The same normalization get_onek1k applies when it builds the DonorData.
donor_key = "OneK1K_" + expr_donors.str.split("_").str[1]
expr_to_geno = dict(zip(expr_donors, donor_key, strict=True))

scored = set(donor_pheno["donor_id"].astype(str))
shared = [d for d in expr_donors if expr_to_geno[d] in scored]
print(f"\n{len(shared):,} of {len(expr_donors):,} expression donors have a PRS")

In [ ]:
donors = rng.choice(np.array(shared), size=min(N_DONORS, len(shared)), replace=False)
top_celltypes = obs[celltype_key].value_counts().head(N_TOP_CELLTYPES)
print(top_celltypes)

eligible = np.where(obs["donor_id"].astype(str).isin(donors) & obs[celltype_key].isin(top_celltypes.index))[0]
keep = np.sort(rng.choice(eligible, size=min(N_CELLS, eligible.size), replace=False))
print(f"\n{eligible.size:,} eligible cells -> keeping {keep.size:,}")

adata = full[keep].to_memory()

# A clean column name, so Step 3's output file is not called
# "step3.predicted.celltype.l2.sceps.omega.celltype.txt".
adata.obs["celltype"] = adata.obs[celltype_key].astype(str)

cells_per_donor = adata.obs["donor_id"].value_counts()
cells_per_donor = cells_per_donor[cells_per_donor > 0]
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes, {cells_per_donor.size} donors")
print(
    f"cells per donor: min {cells_per_donor.min()}, median {int(cells_per_donor.median())}, max {cells_per_donor.max()}"
)

`X` in this file is raw counts, so normalize and log-transform it, drop genes expressed in
almost no cells, and build the k-NN graph **on `X_harmony`** — the embedding the dataset was
published with.

We flag highly variable genes but do *not* subset to them. scEPS needs the full gene set:
its third variance component is "all remaining genes", and control genes are sampled across
the whole expression range.

In [ ]:
print("X max before normalization:", float(adata.X.max()), "(raw counts)" if float(adata.X.max()) > 50 else "")

if float(adata.X.max()) > 50:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

sc.pp.filter_genes(adata, min_cells=10)
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, inplace=True)

# The graph defines the neighborhoods, so it must be built on the batch-harmonized
# embedding -- otherwise neighborhoods track batch rather than biology.
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, use_rep="X_harmony")

print(f"\n{adata.n_obs:,} cells x {adata.n_vars:,} genes retained")
print("k-NN graph:", adata.obsp["connectivities"].shape)
print("embeddings:", list(adata.obsm.keys()))

Attach the phenotype and covariates. scEPS wants them as per-cell columns that are constant
within donor. Covariates named in `--covar` are regressed out of the phenotype up front, so
they must be **donor-level**; the one exception is `sceps.prop_cells_local`, which scEPS
computes per neighborhood and which is worth including so a neighborhood's cell abundance
does not masquerade as a phenotype effect.

In [ ]:
pheno_by_geno_id = donor_pheno.set_index("donor_id")
geno_id = adata.obs["donor_id"].astype(str).map(expr_to_geno)

adata.obs["ra_prs"] = geno_id.map(pheno_by_geno_id["ra_prs_z"]).astype(float)
for i in range(1, 4):
    adata.obs[f"geno_pc{i}"] = geno_id.map(pheno_by_geno_id[f"geno_pc{i}"]).astype(float)
adata.obs["sex"] = adata.obs["sex"].astype(str)

# Donors with a missing phenotype would be dropped by scEPS anyway; do it explicitly.
adata = adata[adata.obs["ra_prs"].notna()].copy()

per_donor = adata.obs.groupby("donor_id", observed=True)[["ra_prs", "geno_pc1"]].first()
print(f"{per_donor.shape[0]} donors with a phenotype")
print(
    f"{PHENO}: mean {per_donor['ra_prs'].mean():.3f}, sd {per_donor['ra_prs'].std():.3f}, "
    f"range {per_donor['ra_prs'].min():.2f} to {per_donor['ra_prs'].max():.2f}"
)
print("sex:", adata.obs.groupby("donor_id", observed=True)["sex"].first().value_counts().to_dict())

In [ ]:
adata.write_h5ad(ADATA_PATH)
print(f"{ADATA_PATH} ({ADATA_PATH.stat().st_size / 1e6:.0f} MB)")
print(adata)

## Step 1: scEPS statistics per cell neighborhood

This is the step that does the work. For each cell in turn, `sceps`:

1. grows a neighborhood around that cell by a random walk on the k-NN graph, expanding until
   it holds enough cells (`--min-num-cell`) and enough donors (`--min-num-donor`,
   `--min-frac-donor`);
2. builds the donor × gene expression matrix for that neighborhood;
3. selects the GWAS genes, and samples control genes matched on mean expression;
4. decomposes the phenotypic variance into GWAS / control / remaining components by method
   of moments, and bootstraps standard errors.

The flags below are the recommended defaults from the wiki:

| Flag | Why |
| --- | --- |
| `--auto-gene-selection` | select GWAS genes by MAGMA FDR rather than a fixed top-N |
| `--scale-pheno` | standardize the phenotype globally before decomposing |
| `--scale-pheno-neighborhood` | standardize again within each neighborhood |
| `--covar` | regress out donor covariates; include `sceps.prop_cells_local` |

### Sizing the run

Cost scales with the number of neighborhoods (one per cell), the number of donors, and the
number of genes. Start with a small probe and measure it on *your* data before committing:
`--start-idx` / `--stop-idx` select a contiguous slice of neighborhoods.

```{note}
scEPS exits without recomputing if its output file already exists, so re-running a cell is
cheap but also does nothing. Pass `--force-rerun` to actually recompute.
```

In [ ]:
!sceps \
    --adata {ADATA_PATH} \
    --donor-id-col donor_id \
    --gene-list {magma_genes} \
    --auto-gene-selection \
    --pheno {PHENO} \
    --scale-pheno \
    --scale-pheno-neighborhood \
    --covar "{COVAR}" \
    --start-idx 0 --stop-idx {N_PROBE} \
    --out {OUT_DIR}/step1

The output file name records the slice — `step1.<start>-<stop>.sceps.omega.txt.gz` —
alongside a `.log` file recording every argument used. The `ELAPSED_TIME` column gives the
per-neighborhood cost, which is what to extrapolate from.

In [ ]:
probe = pd.read_table(OUT_DIR / f"step1.0-{N_PROBE}.sceps.omega.txt.gz")

per_nbhd = probe["ELAPSED_TIME"].mean()
print(f"{per_nbhd:.2f} s per neighborhood")
print(f"-> {N_NEIGHBORHOODS} neighborhoods ~ {per_nbhd * N_NEIGHBORHOODS / 60:.0f} min")
print(f"-> all {adata.n_obs:,} cells ~ {per_nbhd * adata.n_obs / 3600:.1f} h on one core")
print()
print(
    probe[
        ["CELL", "NEIGHBORHOOD_SIZE", "NUM_DONOR", "NUM_GENE_GWAS", "NUM_GENE_CONTROL", "NUM_GENE_REST", "ELAPSED_TIME"]
    ].head()
)

Note `NUM_GENE_GWAS` and `NUM_GENE_CONTROL`: the control set is matched in size to the GWAS
set, and both vary slightly between neighborhoods because genes are filtered per
neighborhood on their across-donor expression variance (`--min-expr-var-thres`,
`--max-expr-var-thres`).

### The tutorial run

Now analyze the rest of the slice. Because the probe already covered neighborhoods 0–20, we
resume at 20 and end up with two output shards — which is exactly how a real, parallelized
run looks. Step 3 combines shards by glob, so nothing extra is needed to stitch them back
together.

In [ ]:
!sceps \
    --adata {ADATA_PATH} \
    --donor-id-col donor_id \
    --gene-list {magma_genes} \
    --auto-gene-selection \
    --pheno {PHENO} \
    --scale-pheno \
    --scale-pheno-neighborhood \
    --covar "{COVAR}" \
    --start-idx {N_PROBE} --stop-idx {N_NEIGHBORHOODS} \
    --out {OUT_DIR}/step1

In [ ]:
print(sorted(p.name for p in OUT_DIR.glob("step1.*")))

The `.log` file is a plain record of the arguments the run used — worth keeping alongside results:

In [ ]:
log = pd.read_table(OUT_DIR / f"step1.0-{N_PROBE}.log")
log[
    log["ARGUMENT"].isin(
        [
            "adata",
            "gene_list",
            "pheno",
            "covar",
            "auto_gene_selection",
            "scale_pheno",
            "scale_pheno_neighborhood",
            "min_frac_donor",
            "num_var_comp",
            "num_bootstrap_regression",
            "seed",
        ]
    )
]

### Reading the output

One row per cell neighborhood, 37 columns. The ones to look at first:

| Column | Meaning |
| --- | --- |
| `CELL` | the cell the neighborhood is anchored on |
| `NEIGHBORHOOD_SIZE`, `NUM_DONOR` | how many cells / donors the neighborhood ended up with |
| `OMEGA_GWAS`, `OMEGA_CONTROL`, `OMEGA_REST` | variance explained per GWAS / control / remaining gene |
| `OMEGA_DIFF` | the $d$ statistic: `OMEGA_GWAS - OMEGA_CONTROL` |
| `SE_OMEGA_DIFF`, `Z_OMEGA_DIFF`, `P_Z_OMEGA_DIFF` | standard error and test of $d \ne 0$ |

The rest are per-neighborhood diagnostics (mean and variance of expression in each gene set,
gene counts, step size, elapsed time) and the same SE/Z/p triplet for each $\omega^2$
component. The
[wiki](https://github.com/Genentech/sceps/wiki/Step-1.-Estimate-scEPS-statistics-at-each-individual-cell-neighborhood)
documents all of them.

In [ ]:
step1 = pd.concat(
    [pd.read_table(p) for p in sorted(OUT_DIR.glob("step1.*.sceps.omega.txt.gz"))],
    ignore_index=True,
)

# Neighborhoods that cannot reach --min-num-cell / --min-num-donor / --min-frac-donor are
# dropped, so fewer rows than requested is normal -- a big shortfall means the QC
# thresholds are too strict for this graph.
print(f"{step1.shape[0]} of {N_NEIGHBORHOODS} neighborhoods passed QC, {step1.shape[1]} columns")
print(
    f"median size {step1['NEIGHBORHOOD_SIZE'].median():.0f} cells, "
    f"{step1['NUM_DONOR'].median():.0f} donors "
    f"(random-walk step size: median {step1['STEP_SIZE'].median():.0f})"
)
print(f"{step1['NUM_GENE_GWAS'].iloc[0]:,} GWAS genes, {step1['NUM_GENE_CONTROL'].iloc[0]:,} control genes")

step1[
    [
        "CELL",
        "NUM_DONOR",
        "NEIGHBORHOOD_SIZE",
        "OMEGA_GWAS",
        "OMEGA_CONTROL",
        "OMEGA_DIFF",
        "SE_OMEGA_DIFF",
        "Z_OMEGA_DIFF",
        "P_Z_OMEGA_DIFF",
    ]
].head()

```{important}
Sign matters. scEPS counts a neighborhood as disease-associated only when $d > 0$ **and**
$\omega^2_{gwas} > 0$ — disease genes must explain *more* than matched controls, and explain
something in absolute terms. A large negative $d$ is not a hit.
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(step1["OMEGA_DIFF"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].axvline(0, color="k", lw=1, ls="--")
axes[0].set(xlabel="$d$  (OMEGA_DIFF)", ylabel="cell neighborhoods", title="Per-neighborhood $d$")

axes[1].scatter(step1["OMEGA_CONTROL"], step1["OMEGA_GWAS"], s=8, alpha=0.5, color="#4C72B0")
lims = [
    min(step1["OMEGA_CONTROL"].min(), step1["OMEGA_GWAS"].min()),
    max(step1["OMEGA_CONTROL"].max(), step1["OMEGA_GWAS"].max()),
]
axes[1].plot(lims, lims, color="k", lw=1, ls="--")
axes[1].set(xlabel=r"$\omega^2_{ctrl}$", ylabel=r"$\omega^2_{gwas}$", title="RA genes vs matched control genes")

plt.tight_layout()
plt.show()

Points above the diagonal are neighborhoods where RA genes out-explain their
expression-matched control genes.

Mapping $d$ onto the embedding is how you would look for structure. We use the UMAP the
dataset ships with, so this is the view the OneK1K authors published:

In [ ]:
adata.obs["sceps_d"] = adata.obs.index.map(step1.set_index("CELL")["OMEGA_DIFF"])
adata.obs["sceps_z"] = adata.obs.index.map(step1.set_index("CELL")["Z_OMEGA_DIFF"])

sc.pl.umap(adata, color="celltype", frameon=False)

# Symmetric colour limits, so the midpoint of the diverging map really is d = 0.
for col in ["sceps_d", "sceps_z"]:
    lim = float(np.nanmax(np.abs(adata.obs[col].to_numpy(dtype=float))))
    sc.pl.umap(adata, color=col, cmap="RdBu_r", vmin=-lim, vmax=lim, na_color="lightgrey", frameon=False)

Grey cells are neighborhoods outside the analyzed slice.

### Scaling up

A real analysis runs every cell neighborhood, which means parallelizing. scEPS offers two
ways, and they produce interchangeable output shards:

```shell
# Option 1: let scEPS split the work evenly across N jobs (array job friendly)
sceps ... --total-num-job 100 --job-idx ${SLURM_ARRAY_TASK_ID} --out out/step1

# Option 2: pick the slice yourself
sceps ... --start-idx 0 --stop-idx 5000 --out out/step1
```

Other options worth knowing when the run is too slow or too large:

| Option | Effect |
| --- | --- |
| `--use-analytical-stderr` | analytical instead of bootstrap standard errors; faster, at the cost of the bootstrap's robustness |
| `--cell-type-col` + `--focal-cell-type` | analyze one cell type at a time, e.g. one job per type |
| `--sample-frac` | down-sample the neighborhoods analyzed |
| `--force-rerun` | recompute even when output exists |
| `--verbose` | per-neighborhood progress |

```{note}
If you split Step 1 by cell type, Step 2 must still be run **once across all cells**. The
blocks it defines are a property of the graph, not of any one analysis.
```

## Step 2: approximately independent neighborhood blocks

Neighborhoods overlap: adjacent cells share cells and donors, so their $d$ statistics are
correlated. Averaging them in Step 3 with a naive bootstrap would treat those correlated
estimates as independent and understate the standard error.

`sceps-cluster-neighborhood` fixes this by mini-batch k-means clustering neighborhoods in
the embedding space, giving blocks of neighborhoods with similar donor composition. Step 3
then resamples **blocks** rather than individual neighborhoods.

Only the graph and the embedding are used, so this can run on a stripped-down `.h5ad` with
no `adata.X`. Two flags matter: `--neighbors-use-rep` names the embedding — the same
harmonized one the graph was built on, here `X_harmony` (the default is `X_pca_harmony`, the
name `misc/preprocess_scdata.py` produces) — and `--num-kmeans-cluster` sets the number of
blocks, whose default of 50 assumes far more cells than this subsample has.

In [ ]:
!sceps-cluster-neighborhood \
    --adata {ADATA_PATH} \
    --donor-id-col donor_id \
    --neighbors-use-rep X_harmony \
    --num-kmeans-cluster {N_KMEANS_CLUSTERS} \
    --out {OUT_DIR}/step2

In [ ]:
blocks = pd.read_table(OUT_DIR / "step2.txt.gz")
print(blocks.head())
print(f"\n{blocks.shape[0]:,} neighborhoods -> {blocks['sceps.neighborhood_cluster'].nunique()} blocks")
print("block sizes:", blocks["sceps.neighborhood_cluster"].value_counts().describe()[["min", "50%", "max"]].to_dict())

The mapping is keyed on `sceps.cell_index` — the cell identifier scEPS derived from
`adata.obs.index`, which is why leaving `--cell-id-col` unset everywhere keeps the steps
consistent.

## Step 3: aggregate to cell types

`sceps-aggregate` averages $d$ across groups of neighborhoods and computes standard errors
by bootstrapping the blocks from Step 2. `--prefix` is a glob over the Step 1 shards, so a
sharded run is combined here. Run it once per grouping you care about:

In [ ]:
# Per cell type
!sceps-aggregate \
    --prefix "{OUT_DIR}/step1.*.sceps.omega.txt.gz" \
    --adata {ADATA_PATH} \
    --neighborhood-clusters {OUT_DIR}/step2.txt.gz \
    --cell-type-col celltype \
    --out {OUT_DIR}/step3

# Across all analyzed neighborhoods (no --cell-type-col)
!sceps-aggregate \
    --prefix "{OUT_DIR}/step1.*.sceps.omega.txt.gz" \
    --adata {ADATA_PATH} \
    --neighborhood-clusters {OUT_DIR}/step2.txt.gz \
    --out {OUT_DIR}/step3

Both runs also write `step3.sceps.omega.txt.gz`: the Step 1 shards concatenated, with
FDR-significance flags added across neighborhoods. Step 4 takes that file as its input.

The key columns mirror Step 1, one level up: `MEAN_OMEGA_DIFF` with `SE_`, `Z_` and `P_Z_`
prefixes. In addition, `NUM_SIGNIF_FDR{5,10,20}_OMEGA_DIFF` count how many *individual*
neighborhoods in the group were significant at each FDR threshold — useful for spotting a
cell type where a minority of neighborhoods carry the signal and the mean washes it out.

In [ ]:
print(sorted(p.name for p in OUT_DIR.glob("step3.*")))

cols = [
    "CELL_TYPE",
    "NUM_CELL",
    "MEAN_NUM_DONOR",
    "MEAN_OMEGA_DIFF",
    "SE_MEAN_OMEGA_DIFF",
    "Z_MEAN_OMEGA_DIFF",
    "P_Z_MEAN_OMEGA_DIFF",
]

by_celltype = pd.read_table(OUT_DIR / "step3.celltype.sceps.omega.celltype.txt")
overall = pd.read_table(OUT_DIR / "step3.sceps.omega.celltype.txt")

print("\n=== by cell type ===")
print(by_celltype[cols].sort_values("Z_MEAN_OMEGA_DIFF", ascending=False).to_string(index=False))
print("\n=== across all analyzed neighborhoods ===")
print(overall[cols].to_string(index=False))

In [ ]:
signif_cols = [
    "CELL_TYPE",
    "NUM_CELL",
    "NUM_SIGNIF_FDR5_OMEGA_DIFF",
    "NUM_SIGNIF_FDR10_OMEGA_DIFF",
    "NUM_SIGNIF_FDR20_OMEGA_DIFF",
]
by_celltype[signif_cols].sort_values("NUM_SIGNIF_FDR20_OMEGA_DIFF", ascending=False)

A forest plot of the per-cell-type estimates is usually the headline figure:

In [ ]:
plot_df = by_celltype.sort_values("MEAN_OMEGA_DIFF")
y = np.arange(plot_df.shape[0])

fig, ax = plt.subplots(figsize=(7.5, 0.5 * plot_df.shape[0] + 1.5))
ax.errorbar(
    plot_df["MEAN_OMEGA_DIFF"],
    y,
    xerr=1.96 * plot_df["SE_MEAN_OMEGA_DIFF"],
    fmt="o",
    color="#4C72B0",
    ecolor="#4C72B0",
    capsize=3,
    lw=1,
)
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_yticks(y)
ax.set_yticklabels([f"{ct}  (n={n:,})" for ct, n in zip(plot_df["CELL_TYPE"], plot_df["NUM_CELL"], strict=True)])
ax.set(
    xlabel="mean $d$  (95% CI, block bootstrap)",
    title="Aggregated scEPS $d$ by cell type\nRA PRS phenotype, RA GWAS genes",
)
plt.tight_layout()
plt.show()

On real data, cell types whose interval sits clearly above zero are the candidate
populations. Keep the PRS caveat from the top of the notebook in mind when reading them.

Other options for this step:

| Option | Effect |
| --- | --- |
| `--cell-type-col a,b,c` | aggregate at several resolutions at once, one output file each |
| `--use-inverse-variance-weights` | weight neighborhoods by precision instead of averaging equally |
| `--use-neighborhoods` | aggregate over an explicit list of cell IDs |
| `--add-column NAME VALUE` | stamp a constant column onto the output, handy when merging many runs |
| `--num-bootstrap` | bootstrap replicates (1,000 by default) |

Omitting `--neighborhood-clusters` falls back to an ordinary bootstrap over individual
neighborhoods, which will generally give standard errors that are too small.

## Step 4 (optional): what distinguishes the associated neighborhoods?

`sceps-corr` correlates each gene's expression with the scEPS statistics across
neighborhoods (Spearman by default). It answers a follow-up question: having found where the
signal is, which genes mark those neighborhoods?

`--min-num-nonzero` drops genes expressed in too few cells. Note that it applies to the
neighborhoods actually analyzed, not the whole atlas — `sceps-corr` intersects the
single-cell data with the scEPS result first — so on a subsampled run it has to come down
from its default of 100.

In [ ]:
!sceps-corr \
    --adata {ADATA_PATH} \
    --sceps-result {OUT_DIR}/step3.sceps.omega.txt.gz \
    --min-num-nonzero {MIN_NUM_NONZERO} \
    --out {OUT_DIR}/step4

In [ ]:
step4 = pd.concat(
    [pd.read_table(p) for p in sorted(OUT_DIR.glob("step4.0-*.txt.gz"))],
    ignore_index=True,
).dropna(subset=["R_OMEGA_DIFF"])

# Map Ensembl IDs back to symbols for readability.
symbols = adata.var["feature_name"].astype(str)
step4["SYMBOL"] = step4["GENE"].map(symbols)

print(f"{step4.shape[0]:,} genes correlated\n")
show = ["SYMBOL", "GENE", "R_OMEGA_GWAS", "R_OMEGA_CONTROL", "R_OMEGA_DIFF"]
print("Most positively correlated with d:")
print(step4.nlargest(10, "R_OMEGA_DIFF")[show].to_string(index=False))
print("\nMost negatively correlated with d:")
print(step4.nsmallest(10, "R_OMEGA_DIFF")[show].to_string(index=False))

The same restricted to one cell type, via `--cell-type-col` / `--focal-cell-type`. With
fewer neighborhoods in a single cell type, `--min-num-nonzero` has to come down further —
otherwise every gene is filtered out and the tool has nothing to concatenate.

In [ ]:
focal = by_celltype.sort_values("Z_MEAN_OMEGA_DIFF", ascending=False)["CELL_TYPE"].iloc[0]
n_focal = int((adata.obs.loc[step1["CELL"], "celltype"] == focal).sum())
focal_min_nonzero = max(5, n_focal // 10)
print(f"focal cell type: {focal} ({n_focal} analyzed neighborhoods) " f"-> --min-num-nonzero {focal_min_nonzero}")

In [ ]:
!sceps-corr \
    --adata {ADATA_PATH} \
    --sceps-result {OUT_DIR}/step3.sceps.omega.txt.gz \
    --cell-type-col celltype \
    --focal-cell-type "{focal}" \
    --min-num-nonzero {focal_min_nonzero} \
    --out {OUT_DIR}/step4_focal

In [ ]:
step4_focal = pd.concat(
    [pd.read_table(p) for p in sorted(OUT_DIR.glob("step4_focal.*.txt.gz"))],
    ignore_index=True,
).dropna(subset=["R_OMEGA_DIFF"])
step4_focal["SYMBOL"] = step4_focal["GENE"].map(symbols)

print(f"{step4_focal.shape[0]:,} genes correlated within {focal}")
step4_focal.nlargest(10, "R_OMEGA_DIFF")[["SYMBOL", "GENE", "R_OMEGA_GWAS", "R_OMEGA_CONTROL", "R_OMEGA_DIFF"]]

```{warning}
By default `sceps-corr` reports correlations with **no** test statistics, deliberately: both
the scEPS statistics and gene expression are correlated across neighborhoods, which makes a
valid test hard to construct. Adding `--num-bootstrap 1000 --neighborhood-clusters
<step2 output>` produces `SE_`, `Z_` and `P_` columns via block bootstrap, but treat those
p-values as descriptive rather than calibrated.
```

## Summary and what changes for a real study

The whole workflow, as four commands:

```shell
# Step 1 -- per-neighborhood statistics (parallelize over --job-idx)
sceps \
    --adata harmonized.h5ad \
    --donor-id-col donor_id \
    --gene-list magma_RA.genes.out \
    --auto-gene-selection \
    --pheno ra_prs \
    --scale-pheno --scale-pheno-neighborhood \
    --covar "sex,geno_pc1,geno_pc2,geno_pc3,sceps.prop_cells_local" \
    --total-num-job 100 --job-idx ${SLURM_ARRAY_TASK_ID} \
    --out out/step1

# Step 2 -- approximately independent blocks (once, across ALL cells)
sceps-cluster-neighborhood \
    --adata harmonized.noX.h5ad \
    --donor-id-col donor_id \
    --neighbors-use-rep X_harmony \
    --out out/step2

# Step 3 -- aggregate to cell types
sceps-aggregate \
    --prefix "out/step1.*.sceps.omega.txt.gz" \
    --adata harmonized.noX.h5ad \
    --neighborhood-clusters out/step2.txt.gz \
    --cell-type-col celltype \
    --out out/step3

# Step 4 -- correlate with gene expression (optional)
sceps-corr \
    --adata harmonized.h5ad \
    --sceps-result out/step3.sceps.omega.txt.gz \
    --out out/step4
```

The wiki suggests one directory per step, which keeps the globs in Steps 3 and 4 simple:

```text
DISEASE/
├── step_0a/out/     # MAGMA gene-level statistics
├── step_0b/out/     # phenotype (PRS, or real disease status)
├── step_0c/out/     # harmonized .h5ad files
├── step1/out/       # one shard per parallel job
├── step2/out/
├── step3/out/
└── step4/out/
```

### Checklist for real data

- **A real disease phenotype** beats a PRS whenever you have one — case/control status or a
  quantitative trait measured in the same donors. scEPS dummy-codes a two-level phenotype
  automatically. If the phenotype lives outside the `.h5ad`, pass it with `--pheno-file`
  (a table keyed by the same donor ID column) rather than merging it in by hand.
- **Add a control arm** when the phenotype *is* a PRS: re-run Step 1 with gene-level
  statistics for an unrelated trait. It costs one extra Step 1 run and is what separates
  trait-specific signal from the genotype→expression confound.
- **Check the gene identifiers match** `adata.var_names` — a mismatch yields an empty gene
  set, not an error.
- **Harmonize before building the graph**, and point `--neighbors-use-rep` at the harmonized
  embedding.
- **Donor-level covariates** in `--covar`, including genotype PCs for a PRS phenotype, and
  keep `sceps.prop_cells_local` in the list.
- **All cells**, via `--total-num-job` / `--job-idx` as an array job.
- **Bootstrap standard errors** (the default). Reach for `--use-analytical-stderr` only when
  compute is the binding constraint.
- **Interpret the sign**: a hit needs $d > 0$ *and* $\omega^2_{gwas} > 0$.

And the caveat once more: nothing above is a finding. OneK1K is a healthy cohort analyzed
with a PRS as a stand-in phenotype, and only a subsample of donors, cells and neighborhoods
was used. See the [wiki](https://github.com/Genentech/sceps/wiki) for the full option
reference and the [preprint](https://www.medrxiv.org/content/10.64898/2026.06.26.26356714v1)
for the method.